# PixAI Core · Colab آزمایشی

این Notebook برای اجرای آزمایشی یک مدل گفت‌وگویی در محیط ابری Google Colab است. مدل روی Runtime ابری دریافت و اجرا می‌شود؛ چیزی روی گوشی نصب نمی‌شود.

**نکته:** این نسخه آزمایشی و موقت است. با قطع شدن Runtime، اجرای مدل متوقف می‌شود و ممکن است لازم باشد مدل دوباره بارگذاری شود. این Notebook یک API دائمی یا سرویس ۲۴ساعته ایجاد نمی‌کند.

مدل پیش‌فرض: `Qwen/Qwen3-1.7B`

## ۱) فعال کردن GPU

در Colab از منوی **Runtime → Change runtime type**، گزینه GPU را انتخاب کن (اگر در حسابت در دسترس بود).

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU در دسترس نیست. برای این مدل، اجرای CPU ممکن است بسیار کند باشد.')

## ۲) نصب کتابخانه‌ها

این مرحله کتابخانه‌های لازم را در Runtime ابری نصب می‌کند.

In [ ]:
%pip -q install -U transformers accelerate

## ۳) بارگذاری مدل

وزن‌های مدل در فضای ابری Colab دریافت می‌شوند. حجم دانلود و زمان بارگذاری به سرعت اینترنت و منابع Runtime بستگی دارد.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

load_kwargs = {'device_map': 'auto'}
if torch.cuda.is_available():
    load_kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print('PixAI Core آماده آزمایش است.')

## ۴) گفت‌وگو با PixAI Core

سلول زیر یک گفت‌وگوی ساده داخل خود Notebook اجرا می‌کند. برای پایان دادن بنویس `خروج` و برای پاک کردن سابقه بنویس `پاک‌کردن`.

## شخصیت، حافظه پایدار و ImageEngine\nدر این نسخه، PixAI دستور هویتی مشخص دارد، تاریخچه چت را در Google Drive نگه می‌دارد و یک اتصال آماده برای ImageEngine دارد. برای فعال‌سازی حافظه باید مجوز Google Drive را بدهی. اتصال تصویر تا وقتی آدرس API معتبر یک سرویس تصویر را تنظیم نکنی غیرفعال می‌ماند.\n\n**حریم خصوصی:** تاریخچه فقط در Google Drive خودت ذخیره می‌شود. پیام‌های حساس، رمزها و اطلاعات محرمانه را وارد چت آزمایشی نکن. فایل حافظه را هر زمان خواستی می‌توانی از Drive پاک کنی.

In [ ]:
SYSTEM_PROMPT = '''تو PixAI Core هستی، دستیار هوش مصنوعی فارسی‌زبان پروژه AIPix.\nهویت و رفتار:\n- اگر کاربر پرسید «کی هستی؟» بگو «من PixAI هستم، دستیار هوش مصنوعی پروژه AIPix».\n- اگر درباره مدل پایه پرسیدند، صادقانه بگو این نسخه بر پایه Qwen اجرا می‌شود. خودت را سازنده یا مالک مدل پایه معرفی نکن.\n- فارسی روان و خودمانی صحبت کن؛ پاسخ‌ها روشن و متناسب با سؤال باشند.\n- دوستانه، صبور، کنجکاو و خلاق باش؛ شوخی سبک مجاز است، بی‌احترامی ممنوع.\n- اگر چیزی را نمی‌دانی، شفاف بگو و حدس را واقعیت جا نزن.\n- ادعا نکن به اینترنت، فایل‌ها، ابزارها یا ImageEngine وصل هستی مگر اتصال واقعاً فعال و موفق باشد.\n- اگر درخواست تصویر شد و ImageEngine فعال نبود، صادقانه توضیح بده که اتصال هنوز تنظیم نشده است.\n- تاریخچه مکالمه‌ای را که در همین نشست بارگذاری شده دنبال کن؛ حافظه بلندمدت فقط از داده‌های ذخیره‌شده در فایل حافظه قابل دسترسی است.\n- اطلاعات خصوصی، رمزها یا کدهای ورود را درخواست نکن.\n- در موضوعات پزشکی، ایمنی و حساس با احتیاط پاسخ بده.\nهدف تو کمک دقیق، امن و مفید به کاربر و پیشبرد پروژه AIPix است.\n'''

## ۵) فعال‌سازی حافظه پایدار در Google Drive\nاین سلول Drive را به‌صورت رسمی mount می‌کند و تاریخچه قبلی را از فایل `AIPix/pixai_memory.json` می‌خواند. بار اول مجوز دسترسی Google را تأیید کن. برای پاک کردن حافظه، متغیر `messages` را ریست کن و فایل را از Google Drive حذف کن.\n

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport json\n\ndrive.mount('/content/drive')\nMEMORY_DIR = Path('/content/drive/MyDrive/AIPix')\nMEMORY_DIR.mkdir(parents=True, exist_ok=True)\nMEMORY_FILE = MEMORY_DIR / 'pixai_memory.json'\n\ndef save_memory():\n    # فقط پیام‌های کاربر و دستیار ذخیره می‌شوند؛ دستور سیستمی ذخیره نمی‌شود.\n    safe_messages = [m for m in messages if m.get('role') in ('user', 'assistant')][-40:]\n    MEMORY_FILE.write_text(json.dumps(safe_messages, ensure_ascii=False, indent=2), encoding='utf-8')\n\nif MEMORY_FILE.exists():\n    try:\n        old_messages = json.loads(MEMORY_FILE.read_text(encoding='utf-8'))\n        if isinstance(old_messages, list):\n            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + [m for m in old_messages if isinstance(m, dict) and m.get('role') in ('user', 'assistant')][-40:]\n        else:\n            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]\n    except Exception as e:\n        print('فایل حافظه خوانده نشد؛ گفت‌وگوی تازه شروع می‌شود:', repr(e))\n        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]\nelse:\n    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]\n\nprint(f'حافظه PixAI آماده است. {len(messages)-1} پیام قبلی بارگذاری شد.')

## ۶) اتصال آماده ImageEngine\nاین بخش یک رابط عمومی برای سرویس‌های تصویرساز HTTP فراهم می‌کند. برای فعال شدن، باید آدرس endpoint سرویس سازگار و در صورت نیاز توکن آن را از خود ارائه‌دهنده بگیری. چون هنوز ارائه‌دهنده و قالب API مشخص نشده، تولید تصویر در این مرحله عمداً غیرفعال است؛ هیچ کلید یا سرویس ساختگی قرار داده نشده است.\n

In [ ]:
import os, getpass, requests, base64\nfrom IPython.display import display, Image as IPyImage\n\nIMAGE_ENGINE_URL = os.environ.get('IMAGE_ENGINE_URL', '').strip()\nIMAGE_ENGINE_TOKEN = os.environ.get('IMAGE_ENGINE_TOKEN', '').strip()\n\ndef configure_image_engine():\n    global IMAGE_ENGINE_URL, IMAGE_ENGINE_TOKEN\n    IMAGE_ENGINE_URL = input('آدرس کامل API تصویرساز را وارد کن: ').strip()\n    IMAGE_ENGINE_TOKEN = getpass.getpass('توکن API (اگر لازم نیست خالی بگذار): ').strip()\n    if not IMAGE_ENGINE_URL.startswith(('https://', 'http://')):\n        raise ValueError('آدرس باید با https:// یا http:// شروع شود.')\n    print('آدرس ImageEngine تنظیم شد. برای امنیت، توکن نمایش داده نمی‌شود.')\n\ndef generate_image(prompt):\n    if not IMAGE_ENGINE_URL:\n        raise RuntimeError('ImageEngine هنوز تنظیم نشده است. ابتدا configure_image_engine() را اجرا کن.')\n    headers = {'Content-Type': 'application/json'}\n    if IMAGE_ENGINE_TOKEN:\n        headers['Authorization'] = f'Bearer {IMAGE_ENGINE_TOKEN}'\n    response = requests.post(IMAGE_ENGINE_URL, headers=headers, json={'prompt': prompt}, timeout=180)\n    response.raise_for_status()\n    data = response.json()\n    item = (data.get('data') or [{}])[0] if isinstance(data.get('data'), list) else {}\n    image_url = data.get('image_url') or data.get('url') or item.get('url')\n    image_b64 = data.get('b64_json') or item.get('b64_json')\n    if image_url:\n        img_response = requests.get(image_url, timeout=120)\n        img_response.raise_for_status()\n        display(IPyImage(data=img_response.content))\n        return 'تصویر از سرویس دریافت شد.'\n    if image_b64:\n        display(IPyImage(data=base64.b64decode(image_b64)))\n        return 'تصویر از سرویس دریافت شد.'\n    raise ValueError('پاسخ API قالب شناخته‌شده‌ای نداشت؛ باید parser را با مستندات سرویس تنظیم کنیم.')\n\nprint('ImageEngine:', 'تنظیم شده' if IMAGE_ENGINE_URL else 'غیرفعال تا زمان وارد کردن endpoint')

## ۷) شروع گفت‌وگو\nپس از اجرای سلول زیر، در همان ورودی با PixAI چت کن. بنویس `خروج` برای پایان، `پاک‌کردن` برای پاک‌کردن تاریخچه ذخیره‌شده، یا `تصویر: ...` برای درخواست تصویر (فقط پس از تنظیم ImageEngine).

In [ ]:
def ask_pixai(user_text, max_new_tokens=384):\n    global messages\n    messages.append({'role': 'user', 'content': user_text})\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)\n    with torch.inference_mode():\n        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.08, pad_token_id=tokenizer.eos_token_id)\n    new_tokens = output[0][inputs['input_ids'].shape[1]:]\n    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()\n    messages.append({'role': 'assistant', 'content': answer})\n    save_memory()\n    return answer\n\nprint('PixAI فعال شد. برای خروج «خروج» را بنویس.')\nwhile True:\n    user_text = input('شما: ').strip()\n    if user_text == 'خروج':\n        save_memory()\n        print('گفت‌وگو ذخیره شد و پایان یافت.')\n        break\n    if user_text == 'پاک‌کردن':\n        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]\n        if MEMORY_FILE.exists(): MEMORY_FILE.unlink()\n        print('حافظه ذخیره‌شده پاک شد.')\n        continue\n    if user_text.startswith('تصویر:'):\n        try: print(generate_image(user_text.split(':',1)[1].strip()))\n        except Exception as e: print('ImageEngine:', repr(e))\n        continue\n    if not user_text: continue\n    try: print('PixAI:', ask_pixai(user_text))\n    except Exception as e:\n        print('خطا:', repr(e))\n        break

## محدودیت‌ها و وضعیت اتصال\n- حافظه پایدار در Google Drive شخصی کاربر ذخیره می‌شود؛ فقط ۴۰ پیام آخر نگه داشته می‌شود.\n- حافظه به معنی آموزش مجدد مدل نیست؛ مدل پایه همچنان Qwen است.\n- ImageEngine فقط وقتی کار می‌کند که endpoint واقعی و سازگار و در صورت نیاز توکن معتبر تنظیم شده باشد. ساختار پاسخ APIهای مختلف ممکن است متفاوت باشد و در آن صورت باید parser را مطابق مستندات همان سرویس تغییر دهیم.\n- Colab رایگان GPU یا اجرای دائمی را تضمین نمی‌کند. با قطع Runtime، مدل باید دوباره بارگذاری شود؛ فایل حافظه در Drive باقی می‌ماند.\n- این Notebook هنوز یک API عمومی یا اپلیکیشن موبایل ۲۴ساعته نیست.